# DAS7002 Big Data Technologies — PRAC 1

## Task 1: Distributed Data Engineering and Pipeline Construction

This notebook implements the data engineering stage of the Chicago Crime analytics project using Apache Spark and PySpark.

The objective of Task 1 is to build a reproducible ETL pipeline that:

- ingests the raw Chicago Crime dataset and supporting datasets;
- applies explicit schema enforcement;
- identifies and handles data quality issues;
- standardises timestamp, location, and categorical fields;
- creates analysis-ready derived attributes;
- prepares supporting socioeconomic, weather, and geographic lookup datasets;
- stores the processed crime data in Parquet format;
- applies an appropriate partitioning strategy using `Year` and `District`.

The processed outputs generated in this notebook will be reused in the exploratory analysis, clustering, and predictive modelling tasks.

## 1. Data Sources

The project uses multiple datasets to support temporal, spatial, socioeconomic, and weather-based analysis.

The main dataset is the Chicago Crimes dataset, containing reported crime incidents from 2001 onward.

Supporting datasets include:

1. **Chicago Crime Records** — the primary large-scale dataset used throughout all tasks.
2. **Selected Socioeconomic Indicators** — community-area-level poverty, income, and hardship measures.
3. **Chicago Community Areas** — community identifiers and geographic boundaries used for lookup and spatial analysis.
4. **NOAA Chicago Midway Weather Data** — daily temperature, precipitation, snowfall, snow depth, and wind measurements for 2001–2023.
5. **Chicago Police District Boundaries** — official police district polygons used later to compare discovered crime clusters against administrative boundaries.

All raw source files are retained unchanged in the project `data/raw` directory. Data cleaning and transformations are performed programmatically using PySpark to preserve reproducibility.

## 2. Spark Environment Setup

Apache Spark is used as the distributed processing engine for the ETL pipeline.

PySpark provides the Python interface to Spark and allows the project to process the large Chicago Crime dataset using distributed DataFrame operations.

The following cell ensures that PySpark 3.5.6 is available in the active local Jupyter runtime.

In [1]:
!pip install pyspark==3.5.6 -q


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 3. Local Runtime and Storage Configuration

This notebook is executed using a local JupyterLab environment on a Windows workstation.

All Python and Apache Spark operations are executed directly on the local machine. Consequently, project datasets are accessed through the local file system using the configured project directory.

Using a persistent local project directory ensures that raw datasets, processed Parquet files, analytical outputs, figures, and model artefacts remain available after the notebook session ends.

The following diagnostic cell records the active Python environment and current working directory to verify the runtime configuration before data ingestion begins.

In [2]:
import os
import sys
import platform

print("Python version :", sys.version.split()[0])
print("Operating system:", platform.system(), platform.release())
print("Working directory:", os.getcwd())

Python version : 3.10.0
Operating system: Windows 10
Working directory: C:\Users\USER\Desktop\Masters\Big Enterprise\DAS7002


## 4. Project Directory Configuration

The project uses a structured local directory to separate raw source files, processed datasets, notebook outputs, figures, tables, and model artefacts.

Centralising the directory paths in one configuration section improves maintainability and reproducibility because subsequent notebook cells can reference consistent variables instead of repeatedly using hard-coded file locations.

In [3]:
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\USER\Desktop\Masters\Big Enterprise\DAS7002")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
MODELS_DIR = PROJECT_ROOT / "outputs" / "models"

print("Project root        :", PROJECT_ROOT)
print("Raw data directory  :", RAW_DIR)
print("Processed directory :", PROCESSED_DIR)
print("Figures directory   :", FIGURES_DIR)
print("Tables directory    :", TABLES_DIR)
print("Models directory    :", MODELS_DIR)

Project root        : C:\Users\USER\Desktop\Masters\Big Enterprise\DAS7002
Raw data directory  : C:\Users\USER\Desktop\Masters\Big Enterprise\DAS7002\data\raw
Processed directory : C:\Users\USER\Desktop\Masters\Big Enterprise\DAS7002\data\processed
Figures directory   : C:\Users\USER\Desktop\Masters\Big Enterprise\DAS7002\outputs\figures
Tables directory    : C:\Users\USER\Desktop\Masters\Big Enterprise\DAS7002\outputs\tables
Models directory    : C:\Users\USER\Desktop\Masters\Big Enterprise\DAS7002\outputs\models


## 5. Raw Data Availability Check

Before constructing the Spark ETL pipeline, the raw-data directory is inspected to confirm that all required source datasets are available and correctly named.

This validation step reduces the risk of downstream processing failures caused by missing files or inconsistent file paths.

In [4]:
raw_files = sorted(RAW_DIR.iterdir())

print(f"Number of files found: {len(raw_files)}")
print()

for file_path in raw_files:
    print(file_path.name)

Number of files found: 7

community_areas.csv
community_areas.geojson
Crimes_-_2001_to_Present.csv
police_districts.csv
police_districts.geojson
socioeconomic.csv
weather_midway_2001_2023.csv


### 5.1 Required Dataset Validation

The following check verifies that each dataset required by the project exists in the raw-data directory before the ETL process begins.

In [5]:
required_files = [
    "Crimes_-_2001_to_Present.csv",
    "socioeconomic.csv",
    "community_areas.csv",
    "community_areas.geojson",
    "weather_midway_2001_2023.csv",
    "police_districts.csv",
    "police_districts.geojson"
]

missing_files = [
    file_name
    for file_name in required_files
    if not (RAW_DIR / file_name).exists()
]

if missing_files:
    print("Missing required files:")
    for file_name in missing_files:
        print(f" - {file_name}")
else:
    print("Validation passed: all required raw datasets are available.")

Validation passed: all required raw datasets are available.


## 5.2 Windows Hadoop Configuration

Because this notebook runs Apache Spark locally on Windows, Hadoop's local filesystem helper is configured before the Spark JVM is initialized.

The `HADOOP_HOME` environment variable points to the local Hadoop utility directory containing `winutils.exe`.

In [6]:
import os
import shutil

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))
print("winutils path:", shutil.which("winutils"))

HADOOP_HOME: C:\hadoop
winutils path: C:\hadoop\bin\winutils.EXE


## 6. Apache Spark Session Initialization

Apache Spark is used as the distributed processing engine for the ETL pipeline.

A Spark session is initialized below to enable distributed DataFrame operations on the Chicago Crime dataset and the supporting datasets. The configuration uses a small number of shuffle partitions because the notebook is running on a local workstation rather than a multi-node cluster.

The runtime details are printed to confirm that Spark has initialized successfully and to document the execution environment used for the analysis.

In [7]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("DAS7002_PRAC1_ChicagoCrime")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Spark session created successfully.")
print("Spark version:", spark.version)
print("Application name:", spark.sparkContext.appName)
print("Master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Spark session created successfully.
Spark version: 3.5.6
Application name: DAS7002_PRAC1_ChicagoCrime
Master: local[*]
Default parallelism: 8
Shuffle partitions: 8


### 6.1 Runtime Verification

The Spark session initialized successfully using the local execution environment.

The `local[*]` master setting allows Spark to use the available logical processor cores on the workstation. A relatively small shuffle partition count is used to avoid unnecessary task overhead during local execution. This value can be adjusted later if performance profiling indicates that a different level of parallelism would be more appropriate.

## 7. Chicago Crime Dataset Schema Enforcement

The Chicago Crime dataset contains a mixture of identifiers, categorical attributes, Boolean fields, numeric coordinates, and date/time values.

Rather than relying on automatic schema inference, an explicit `StructType` schema is defined before ingestion. This provides stronger control over data types, reduces ambiguity, and helps identify malformed or inconsistent values during the ETL process.

Date fields are initially loaded as strings because the source uses a custom timestamp format. These fields will be parsed and validated explicitly in the subsequent cleaning stage.

In [8]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    IntegerType,
    DoubleType,
    BooleanType
)

crime_schema = StructType([
    StructField("ID", LongType(), True),
    StructField("Case Number", StringType(), True),
    StructField("Date", StringType(), True),
    StructField("Block", StringType(), True),
    StructField("IUCR", StringType(), True),
    StructField("Primary Type", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Location Description", StringType(), True),
    StructField("Arrest", BooleanType(), True),
    StructField("Domestic", BooleanType(), True),
    StructField("Beat", IntegerType(), True),
    StructField("District", IntegerType(), True),
    StructField("Ward", IntegerType(), True),
    StructField("Community Area", IntegerType(), True),
    StructField("FBI Code", StringType(), True),
    StructField("X Coordinate", DoubleType(), True),
    StructField("Y Coordinate", DoubleType(), True),
    StructField("Year", IntegerType(), True),
    StructField("Updated On", StringType(), True),
    StructField("Latitude", DoubleType(), True),
    StructField("Longitude", DoubleType(), True),
    StructField("Location", StringType(), True)
])

print("Explicit crime schema created successfully.")
print("Number of fields:", len(crime_schema.fields))

Explicit crime schema created successfully.
Number of fields: 22


### 7.1 Schema Design Rationale

Several schema decisions are intentional:

- `ID` is represented as a long integer because it is a large numerical identifier.
- `IUCR` and `FBI Code` are stored as strings because these fields may contain leading zeros or non-numeric codes.
- `Arrest` and `Domestic` are represented as Boolean values.
- `Latitude`, `Longitude`, and coordinate fields are represented as double-precision numeric values.
- `Date` and `Updated On` are initially retained as strings so that timestamp parsing can be performed explicitly using the known source format.
- Nullable fields are permitted because missing values are expected in several columns, particularly geographic attributes.

The explicit schema therefore acts as an initial data-quality control while preserving potentially valid records that contain incomplete attributes.

## 8. Chicago Crime Data Ingestion

The raw Chicago Crime CSV is now ingested using the explicit schema defined in the previous section.

Using an explicit schema avoids the cost and uncertainty of automatic type inference and ensures that Spark interprets the dataset consistently from the beginning of the ETL pipeline.

The source file is read with the header enabled, while malformed or incomplete values are preserved as nulls for later data-quality assessment rather than being silently discarded.

In [9]:
CRIME_PATH = RAW_DIR / "Crimes_-_2001_to_Present.csv"

crime_df = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .schema(crime_schema)
    .csv(str(CRIME_PATH))
)

print("Chicago Crime dataset loaded successfully.")
print("Number of columns:", len(crime_df.columns))

Chicago Crime dataset loaded successfully.
Number of columns: 22


### 8.1 Loaded Schema Verification

The loaded DataFrame schema is inspected to confirm that the explicit schema has been applied correctly to all 22 source attributes.

In [10]:
crime_df.printSchema()

root
 |-- ID: long (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community Area: integer (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: double (nullable = true)
 |-- Y Coordinate: double (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)



### 8.2 Initial Record Inspection

A small sample of raw records is displayed to verify that values have been loaded into the expected columns before any cleaning or transformation is performed.

Only a limited number of records are displayed because the full dataset is large and should remain distributed within Spark.

In [11]:
crime_df.show(5, truncate=False)

+--------+-----------+----------------------+---------------------+----+------------+-----------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+------------+-------------+-----------------------------+
|ID      |Case Number|Date                  |Block                |IUCR|Primary Type|Description            |Location Description|Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On            |Latitude    |Longitude    |Location                     |
+--------+-----------+----------------------+---------------------+----+------------+-----------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+------------+-------------+-----------------------------+
|10224738|HY411648   |09/05/2015 01:30:00 PM|043XX S WOOD ST      |0486|BATTERY     |DOMESTIC BAT

## 9. Initial Data Quality Profiling

Before applying any cleaning rules, the raw Chicago Crime DataFrame is profiled to establish a baseline view of data completeness and quality.

The profiling stage records:

- total row count;
- duplicate record count;
- null values across important analytical fields;
- minimum and maximum year values;
- minimum and maximum crime timestamps;
- missing geographic coordinates;
- missing district and community-area identifiers.

These baseline metrics will later be compared against the cleaned dataset to demonstrate the effect of the ETL process.

In [12]:
from pyspark.sql import functions as F

total_rows = crime_df.count()

print(f"Total raw crime records: {total_rows:,}")

Total raw crime records: 7,784,664


### 9.1 Null Value Profiling

Null values are measured for fields that are particularly important for temporal, spatial, and predictive analysis.

Rather than immediately deleting incomplete records, the extent of missingness is measured first so that subsequent cleaning decisions can be justified.

In [13]:
from pyspark.sql import functions as F

important_columns = [
    "ID",
    "Case Number",
    "Date",
    "Primary Type",
    "Location Description",
    "District",
    "Ward",
    "Community Area",
    "Year",
    "Latitude",
    "Longitude",
    "Location"
]

# Perform a single Spark aggregation for the total row count
# and null counts across all selected fields.
aggregation_expressions = [
    F.count("*").alias("Total_Rows")
]

for i, column_name in enumerate(important_columns):
    aggregation_expressions.append(
        F.sum(
            F.col(column_name).isNull().cast("long")
        ).alias(f"null_{i}")
    )

profile_wide_df = crime_df.agg(*aggregation_expressions)

# Convert the one-row wide result into a readable long-format table.
stack_expression = ", ".join(
    [
        f"'{column_name}', null_{i}"
        for i, column_name in enumerate(important_columns)
    ]
)

null_profile_df = (
    profile_wide_df
    .selectExpr(
        "Total_Rows",
        f"stack({len(important_columns)}, {stack_expression}) "
        "as (Column, Null_Count)"
    )
    .withColumn(
        "Null_Percentage",
        F.round(
            (F.col("Null_Count") / F.col("Total_Rows")) * 100,
            4
        )
    )
    .select(
        "Column",
        "Null_Count",
        "Null_Percentage"
    )
)

null_profile_df.show(n=len(important_columns), truncate=False)

+--------------------+----------+---------------+
|Column              |Null_Count|Null_Percentage|
+--------------------+----------+---------------+
|ID                  |0         |0.0            |
|Case Number         |4         |1.0E-4         |
|Date                |0         |0.0            |
|Primary Type        |0         |0.0            |
|Location Description|10381     |0.1334         |
|District            |47        |6.0E-4         |
|Ward                |614848    |7.8982         |
|Community Area      |613476    |7.8806         |
|Year                |0         |0.0            |
|Latitude            |86848     |1.1156         |
|Longitude           |86848     |1.1156         |
|Location            |86848     |1.1156         |
+--------------------+----------+---------------+



### 9.2 Initial Missing-Data Observation

The baseline profile shows that the core crime identifiers and temporal attributes are highly complete, with no missing values observed for `ID`, `Date`, `Primary Type`, or `Year`.

Missingness is more pronounced in selected geographic and administrative attributes. In particular, `Ward` and `Community Area` contain substantially more missing values than `District`, while latitude, longitude, and the combined location field share the same number of missing observations.

These differences are important because the analytical tasks rely on geographic information at different levels of granularity. Records will therefore not be removed solely because one optional geographic attribute is missing. Instead, cleaning rules will be defined according to the requirements of each downstream analysis.

### 9.3 Duplicate Record Assessment

The `ID` field is expected to uniquely identify individual crime records. Duplicate identifiers are therefore checked before any deduplication rule is applied.

A distinction is made between duplicate identifiers and exact duplicate rows because repeated identifiers may indicate data-quality issues that require separate investigation.

In [14]:
duplicate_id_df = (
    crime_df
    .groupBy("ID")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_id_count = duplicate_id_df.count()

print(f"Number of duplicated ID values: {duplicate_id_count:,}")

Number of duplicated ID values: 0


In [15]:
if duplicate_id_count > 0:
    duplicate_id_df.orderBy(F.desc("count")).show(10, truncate=False)
else:
    print("No duplicated ID values were found.")

No duplicated ID values were found.


### 9.4 Exact Duplicate Row Assessment

In addition to identifier-level duplication, the dataset is checked for completely duplicated rows.

Exact row duplication can unnecessarily increase storage and computation costs and may bias aggregate crime counts if retained.

In [16]:
distinct_row_count = crime_df.distinct().count()
exact_duplicate_rows = total_rows - distinct_row_count

print(f"Total raw records      : {total_rows:,}")
print(f"Distinct records       : {distinct_row_count:,}")
print(f"Exact duplicate records: {exact_duplicate_rows:,}")

Total raw records      : 7,784,664
Distinct records       : 7,784,664
Exact duplicate records: 0


### 9.5 Temporal Range Validation

The source `Date` field is stored as a string and must therefore be explicitly parsed before temporal analysis.

The timestamp format used by the source is `MM/dd/yyyy hh:mm:ss a`. Parsing the field explicitly allows invalid timestamps to be identified and supports reliable calculation of the dataset's temporal coverage.

In [17]:
crime_temporal_check_df = (
    crime_df
    .withColumn(
        "Parsed_Date",
        F.to_timestamp(
            F.col("Date"),
            "MM/dd/yyyy hh:mm:ss a"
        )
    )
)

temporal_summary = crime_temporal_check_df.agg(
    F.min("Year").alias("Minimum_Year"),
    F.max("Year").alias("Maximum_Year"),
    F.min("Parsed_Date").alias("Earliest_Crime_Date"),
    F.max("Parsed_Date").alias("Latest_Crime_Date"),
    F.sum(
        F.col("Parsed_Date").isNull().cast("long")
    ).alias("Invalid_Parsed_Dates")
)

temporal_summary.show(truncate=False)

+------------+------------+-------------------+-------------------+--------------------+
|Minimum_Year|Maximum_Year|Earliest_Crime_Date|Latest_Crime_Date  |Invalid_Parsed_Dates|
+------------+------------+-------------------+-------------------+--------------------+
|2001        |2023        |2001-01-01 00:00:00|2023-04-21 23:59:00|0                   |
+------------+------------+-------------------+-------------------+--------------------+



## 10. Data Cleaning Strategy

The initial profiling stage showed that the Chicago Crime dataset is structurally consistent, contains no duplicate identifiers or exact duplicate rows, and has fully valid crime timestamps across the 2001–2023 period.

Consequently, the cleaning strategy focuses on targeted quality improvements rather than aggressive row deletion.

The main cleaning actions are:

- parse the `Date` and `Updated On` fields into Spark timestamp types;
- retain records with missing optional geographic attributes rather than removing them globally;
- validate latitude and longitude values against plausible Chicago-area ranges;
- preserve `District`, `Ward`, and `Community Area` as nullable administrative attributes;
- standardise selected string fields by trimming unnecessary whitespace;
- create derived temporal attributes required for later analysis;
- retain all valid crime records unless a field is specifically required by a downstream analytical task.

This approach minimises unnecessary information loss while ensuring that analytical subsets can be created safely for spatial, temporal, and predictive tasks.

In [18]:
crime_clean_df = (
    crime_df

    # Parse timestamp fields
    .withColumn(
        "Crime_Timestamp",
        F.to_timestamp(
            F.col("Date"),
            "MM/dd/yyyy hh:mm:ss a"
        )
    )
    .withColumn(
        "Updated_Timestamp",
        F.to_timestamp(
            F.col("Updated On"),
            "MM/dd/yyyy hh:mm:ss a"
        )
    )

    # Standardise selected text fields
    .withColumn(
        "Primary Type",
        F.trim(F.col("Primary Type"))
    )
    .withColumn(
        "Description",
        F.trim(F.col("Description"))
    )
    .withColumn(
        "Location Description",
        F.trim(F.col("Location Description"))
    )
    .withColumn(
        "Block",
        F.trim(F.col("Block"))
    )
)

### 10.1 Derived Temporal Attributes

Several temporal attributes are derived from the parsed crime timestamp to support later exploratory analysis and spatial-temporal clustering.

These derived features include:

- calendar date;
- year;
- month;
- month name;
- day of month;
- day of week;
- hour of day;
- weekend indicator.

The derived fields are created using Spark SQL functions so that the transformations remain distributed and reproducible.

In [19]:
crime_clean_df = (
    crime_clean_df

    .withColumn(
        "Crime_Date",
        F.to_date(F.col("Crime_Timestamp"))
    )
    .withColumn(
        "Crime_Year",
        F.year(F.col("Crime_Timestamp"))
    )
    .withColumn(
        "Crime_Month",
        F.month(F.col("Crime_Timestamp"))
    )
    .withColumn(
        "Crime_Month_Name",
        F.date_format(F.col("Crime_Timestamp"), "MMMM")
    )
    .withColumn(
        "Crime_Day",
        F.dayofmonth(F.col("Crime_Timestamp"))
    )
    .withColumn(
        "Crime_Day_Of_Week",
        F.date_format(F.col("Crime_Timestamp"), "EEEE")
    )
    .withColumn(
        "Crime_Hour",
        F.hour(F.col("Crime_Timestamp"))
    )
    .withColumn(
        "Is_Weekend",
        F.when(
            F.dayofweek(F.col("Crime_Timestamp")).isin([1, 7]),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
)

### 10.2 Transformation Verification

A small sample is displayed to verify that the original timestamp has been parsed correctly and that the derived temporal attributes are consistent with the source record.

In [20]:
crime_clean_df.select(
    "ID",
    "Date",
    "Crime_Timestamp",
    "Crime_Date",
    "Crime_Year",
    "Crime_Month",
    "Crime_Month_Name",
    "Crime_Day_Of_Week",
    "Crime_Hour",
    "Is_Weekend"
).show(10, truncate=False)

+--------+----------------------+-------------------+----------+----------+-----------+----------------+-----------------+----------+----------+
|ID      |Date                  |Crime_Timestamp    |Crime_Date|Crime_Year|Crime_Month|Crime_Month_Name|Crime_Day_Of_Week|Crime_Hour|Is_Weekend|
+--------+----------------------+-------------------+----------+----------+-----------+----------------+-----------------+----------+----------+
|10224738|09/05/2015 01:30:00 PM|2015-09-05 13:30:00|2015-09-05|2015      |9          |September       |Saturday         |13        |true      |
|10224739|09/04/2015 11:30:00 AM|2015-09-04 11:30:00|2015-09-04|2015      |9          |September       |Friday           |11        |false     |
|11646166|09/01/2018 12:01:00 AM|2018-09-01 00:01:00|2018-09-01|2018      |9          |September       |Saturday         |0         |true      |
|10224740|09/05/2015 12:45:00 PM|2015-09-05 12:45:00|2015-09-05|2015      |9          |September       |Saturday         |12      

## 11. Geographic Data Validation

Spatial analysis in later tasks depends on reliable latitude and longitude values.

The initial null profile showed that approximately 1.12% of records have missing geographic coordinates. Before creating spatial subsets, the coordinate values that are present are checked against plausible geographic bounds for the Chicago area.

This validation distinguishes between:

- missing coordinates;
- coordinates outside plausible Chicago-area bounds;
- valid spatial records suitable for clustering and mapping.

Records are not removed from the main cleaned dataset solely because geographic coordinates are unavailable. Instead, a separate spatially valid subset will be created for analyses that specifically require location data.

In [21]:
# Plausible bounding box covering the Chicago metropolitan area.
CHICAGO_LAT_MIN = 41.60
CHICAGO_LAT_MAX = 42.10
CHICAGO_LON_MIN = -88.00
CHICAGO_LON_MAX = -87.50

geo_profile_df = crime_clean_df.agg(
    F.sum(
        (
            F.col("Latitude").isNull() |
            F.col("Longitude").isNull()
        ).cast("long")
    ).alias("Missing_Coordinates"),

    F.sum(
        (
            F.col("Latitude").isNotNull() &
            F.col("Longitude").isNotNull() &
            (
                (F.col("Latitude") < CHICAGO_LAT_MIN) |
                (F.col("Latitude") > CHICAGO_LAT_MAX) |
                (F.col("Longitude") < CHICAGO_LON_MIN) |
                (F.col("Longitude") > CHICAGO_LON_MAX)
            )
        ).cast("long")
    ).alias("Out_Of_Bounds_Coordinates"),

    F.sum(
        (
            F.col("Latitude").between(
                CHICAGO_LAT_MIN,
                CHICAGO_LAT_MAX
            ) &
            F.col("Longitude").between(
                CHICAGO_LON_MIN,
                CHICAGO_LON_MAX
            )
        ).cast("long")
    ).alias("Valid_Coordinates")
)

geo_profile_df.show(truncate=False)

+-------------------+-------------------------+-----------------+
|Missing_Coordinates|Out_Of_Bounds_Coordinates|Valid_Coordinates|
+-------------------+-------------------------+-----------------+
|86848              |118                      |7697698          |
+-------------------+-------------------------+-----------------+



### 11.1 Spatially Valid Analytical Subset

A separate DataFrame is created for analyses requiring geographic coordinates.

This preserves the complete cleaned crime dataset for non-spatial analysis while ensuring that clustering and mapping operations use only records with valid latitude and longitude values.

In [22]:
crime_spatial_df = (
    crime_clean_df
    .filter(
        F.col("Latitude").between(
            CHICAGO_LAT_MIN,
            CHICAGO_LAT_MAX
        ) &
        F.col("Longitude").between(
            CHICAGO_LON_MIN,
            CHICAGO_LON_MAX
        )
    )
)

spatial_row_count = crime_spatial_df.count()

print(f"Total cleaned records      : {total_rows:,}")
print(f"Spatially valid records    : {spatial_row_count:,}")
print(
    f"Spatial retention rate     : "
    f"{(spatial_row_count / total_rows) * 100:.4f}%"
)

Total cleaned records      : 7,784,664
Spatially valid records    : 7,697,698
Spatial retention rate     : 98.8829%


## 12. Socioeconomic Dataset Ingestion and Schema Enforcement

The socioeconomic dataset provides community-area-level indicators that can later be joined to crime records using the community area identifier.

An explicit schema is applied to ensure numeric measures such as poverty percentage, per-capita income, and hardship index are interpreted consistently.

The final citywide aggregate row is not an individual community area and will therefore be excluded from community-level joins.

In [23]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType
)

socio_schema = StructType([
    StructField("Community Area Number", IntegerType(), True),
    StructField("COMMUNITY AREA NAME", StringType(), True),
    StructField("PERCENT HOUSEHOLDS BELOW POVERTY", DoubleType(), True),
    StructField("PER CAPITA INCOME ", IntegerType(), True),
    StructField("HARDSHIP INDEX", IntegerType(), True)
])

SOCIO_PATH = RAW_DIR / "socioeconomic.csv"

socio_df = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .schema(socio_schema)
    .csv(str(SOCIO_PATH))
)

print("Socioeconomic dataset loaded successfully.")
print("Number of columns:", len(socio_df.columns))

Socioeconomic dataset loaded successfully.
Number of columns: 5


### 12.1 Socioeconomic Schema and Record Inspection

The schema and a sample of records are inspected to confirm that community identifiers and numerical indicators have been loaded correctly before cleaning.

In [24]:
socio_df.printSchema()

root
 |-- Community Area Number: integer (nullable = true)
 |-- COMMUNITY AREA NAME: string (nullable = true)
 |-- PERCENT HOUSEHOLDS BELOW POVERTY: double (nullable = true)
 |-- PER CAPITA INCOME : integer (nullable = true)
 |-- HARDSHIP INDEX: integer (nullable = true)



In [25]:
socio_df.show(10, truncate=False)

+---------------------+-------------------+--------------------------------+------------------+--------------+
|Community Area Number|COMMUNITY AREA NAME|PERCENT HOUSEHOLDS BELOW POVERTY|PER CAPITA INCOME |HARDSHIP INDEX|
+---------------------+-------------------+--------------------------------+------------------+--------------+
|1                    |Rogers Park        |23.6                            |23939             |39            |
|2                    |West Ridge         |17.2                            |23040             |46            |
|3                    |Uptown             |24.0                            |35787             |20            |
|4                    |Lincoln Square     |10.9                            |37524             |17            |
|5                    |North Center       |7.5                             |57123             |6             |
|6                    |Lake View          |11.4                            |60058             |5             |
|

### 12.2 Community-Level Record Validation

The Chicago community-area system contains 77 numbered community areas.

The source dataset also contains a citywide Chicago aggregate row without a community area number. This record is useful as a reference summary but should not be included in community-level joins with crime records.

In [26]:
socio_total_rows = socio_df.count()

invalid_socio_rows = (
    socio_df
    .filter(F.col("Community Area Number").isNull())
    .count()
)

print(f"Total socioeconomic records          : {socio_total_rows}")
print(f"Rows without community area number   : {invalid_socio_rows}")

Total socioeconomic records          : 78
Rows without community area number   : 1


### 12.3 Socioeconomic Data Cleaning

The citywide aggregate row is excluded so that the resulting dataset contains only the 77 numbered community areas.

Column names are also standardised to improve readability and simplify later joins.

In [27]:
socio_clean_df = (
    socio_df
    .filter(F.col("Community Area Number").isNotNull())
    .select(
        F.col("Community Area Number").alias("Community_Area"),
        F.trim(F.col("COMMUNITY AREA NAME")).alias("Community_Name"),
        F.col("PERCENT HOUSEHOLDS BELOW POVERTY").alias("Poverty_Percent"),
        F.col("PER CAPITA INCOME ").alias("Per_Capita_Income"),
        F.col("HARDSHIP INDEX").alias("Hardship_Index")
    )
)

In [28]:
print("Clean socioeconomic records:", socio_clean_df.count())

socio_clean_df.show(10, truncate=False)

Clean socioeconomic records: 77
+--------------+---------------+---------------+-----------------+--------------+
|Community_Area|Community_Name |Poverty_Percent|Per_Capita_Income|Hardship_Index|
+--------------+---------------+---------------+-----------------+--------------+
|1             |Rogers Park    |23.6           |23939            |39            |
|2             |West Ridge     |17.2           |23040            |46            |
|3             |Uptown         |24.0           |35787            |20            |
|4             |Lincoln Square |10.9           |37524            |17            |
|5             |North Center   |7.5            |57123            |6             |
|6             |Lake View      |11.4           |60058            |5             |
|7             |Lincoln Park   |12.3           |71551            |2             |
|8             |Near North Side|12.9           |88669            |1             |
|9             |Edison Park    |3.3            |40959            |

### 12.4 Community Identifier Validation

The cleaned socioeconomic dataset is checked to confirm that community area identifiers span the expected range from 1 to 77.

In [29]:
socio_clean_df.agg(
    F.min("Community_Area").alias("Minimum_Community_Area"),
    F.max("Community_Area").alias("Maximum_Community_Area"),
    F.countDistinct("Community_Area").alias("Distinct_Community_Areas")
).show()

+----------------------+----------------------+------------------------+
|Minimum_Community_Area|Maximum_Community_Area|Distinct_Community_Areas|
+----------------------+----------------------+------------------------+
|                     1|                    77|                      77|
+----------------------+----------------------+------------------------+



## 13. Community Area Lookup Dataset

The Chicago Community Areas dataset contains geographic boundary information together with community-area identifiers and names.

For the later Broadcast join requirement, only the small lookup attributes are required. Therefore, the large geometry column is not retained in the lookup DataFrame.

Reducing the dataset to community area number and community name produces a compact dimension-style table that can be efficiently broadcast during joins with the much larger crime dataset.

In [30]:
community_schema = StructType([
    StructField("the_geom", StringType(), True),
    StructField("AREA_NUMBE", IntegerType(), True),
    StructField("COMMUNITY", StringType(), True),
    StructField("AREA_NUM_1", IntegerType(), True),
    StructField("SHAPE_AREA", StringType(), True),
    StructField("SHAPE_LEN", StringType(), True)
])

COMMUNITY_CSV_PATH = RAW_DIR / "community_areas.csv"

community_df = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .schema(community_schema)
    .csv(str(COMMUNITY_CSV_PATH))
)

print("Community Areas dataset loaded successfully.")
print("Number of columns:", len(community_df.columns))

Community Areas dataset loaded successfully.
Number of columns: 6


### 13.1 Community Area Source Validation

The source structure is inspected before reducing the dataset to the attributes required for lookup operations.

In [31]:
community_df.printSchema()

root
 |-- the_geom: string (nullable = true)
 |-- AREA_NUMBE: integer (nullable = true)
 |-- COMMUNITY: string (nullable = true)
 |-- AREA_NUM_1: integer (nullable = true)
 |-- SHAPE_AREA: string (nullable = true)
 |-- SHAPE_LEN: string (nullable = true)



In [32]:
community_df.select(
    "AREA_NUMBE",
    "COMMUNITY",
    "AREA_NUM_1"
).show(10, truncate=False)

+----------+---------------+----------+
|AREA_NUMBE|COMMUNITY      |AREA_NUM_1|
+----------+---------------+----------+
|1         |ROGERS PARK    |1         |
|2         |WEST RIDGE     |2         |
|3         |UPTOWN         |3         |
|4         |LINCOLN SQUARE |4         |
|5         |NORTH CENTER   |5         |
|6         |LAKE VIEW      |6         |
|7         |LINCOLN PARK   |7         |
|8         |NEAR NORTH SIDE|8         |
|9         |EDISON PARK    |9         |
|10        |NORWOOD PARK   |10        |
+----------+---------------+----------+
only showing top 10 rows



### 13.2 Compact Community Lookup Construction

Only the community identifier and community name are retained for lookup operations.

The community name is trimmed to remove unnecessary whitespace, and the resulting table is deduplicated by community area number to ensure that each identifier maps to one community name.

In [33]:
community_lookup_df = (
    community_df
    .select(
        F.col("AREA_NUMBE").alias("Community_Area"),
        F.trim(F.col("COMMUNITY")).alias("Community_Name")
    )
    .filter(F.col("Community_Area").isNotNull())
    .dropDuplicates(["Community_Area"])
    .orderBy("Community_Area")
)

In [34]:
community_lookup_count = community_lookup_df.count()

print(f"Community lookup records: {community_lookup_count}")

community_lookup_df.show(10, truncate=False)

Community lookup records: 77
+--------------+---------------+
|Community_Area|Community_Name |
+--------------+---------------+
|1             |ROGERS PARK    |
|2             |WEST RIDGE     |
|3             |UPTOWN         |
|4             |LINCOLN SQUARE |
|5             |NORTH CENTER   |
|6             |LAKE VIEW      |
|7             |LINCOLN PARK   |
|8             |NEAR NORTH SIDE|
|9             |EDISON PARK    |
|10            |NORWOOD PARK   |
+--------------+---------------+
only showing top 10 rows



### 13.3 Cross-Dataset Community Identifier Validation

The community lookup identifiers are compared with the socioeconomic dataset to confirm that both sources describe the same 77 community areas.

This check reduces the risk of join mismatches in later analytical stages.

In [35]:
lookup_only_ids = (
    community_lookup_df
    .select("Community_Area")
    .subtract(
        socio_clean_df.select("Community_Area")
    )
)

socio_only_ids = (
    socio_clean_df
    .select("Community_Area")
    .subtract(
        community_lookup_df.select("Community_Area")
    )
)

lookup_only_count = lookup_only_ids.count()
socio_only_count = socio_only_ids.count()

print("Community IDs only in lookup dataset :", lookup_only_count)
print("Community IDs only in socioeconomic   :", socio_only_count)

Community IDs only in lookup dataset : 0
Community IDs only in socioeconomic   : 0


## 14. Weather Dataset Ingestion and Schema Enforcement

Daily weather observations from Chicago Midway Airport are used to support the later analysis of weather conditions and crime frequency.

The dataset contains daily observations for temperature, precipitation, snowfall, snow depth, and average wind speed between 2001 and 2023.

An explicit schema is applied so that the date and weather measurements are interpreted consistently before cleaning and temporal joining.

In [36]:
weather_schema = StructType([
    StructField("STATION", StringType(), True),
    StructField("NAME", StringType(), True),
    StructField("DATE", StringType(), True),
    StructField("AWND", DoubleType(), True),
    StructField("PRCP", DoubleType(), True),
    StructField("SNOW", DoubleType(), True),
    StructField("SNWD", DoubleType(), True),
    StructField("TMAX", DoubleType(), True),
    StructField("TMIN", DoubleType(), True)
])

WEATHER_PATH = RAW_DIR / "weather_midway_2001_2023.csv"

weather_df = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .schema(weather_schema)
    .csv(str(WEATHER_PATH))
)

print("Weather dataset loaded successfully.")
print("Number of columns:", len(weather_df.columns))

Weather dataset loaded successfully.
Number of columns: 9


### 14.1 Weather Schema and Record Inspection

The loaded schema and sample records are inspected to verify that numeric weather measurements and the source date field have been interpreted correctly.

In [37]:
weather_df.printSchema()

root
 |-- STATION: string (nullable = true)
 |-- NAME: string (nullable = true)
 |-- DATE: string (nullable = true)
 |-- AWND: double (nullable = true)
 |-- PRCP: double (nullable = true)
 |-- SNOW: double (nullable = true)
 |-- SNWD: double (nullable = true)
 |-- TMAX: double (nullable = true)
 |-- TMIN: double (nullable = true)



In [38]:
weather_df.show(10, truncate=False)

+-----------+-----------------------------+----------+-----+----+----+----+----+----+
|STATION    |NAME                         |DATE      |AWND |PRCP|SNOW|SNWD|TMAX|TMIN|
+-----------+-----------------------------+----------+-----+----+----+----+----+----+
|USW00014819|CHICAGO MIDWAY AIRPORT, IL US|2001-01-01|7.38 |0.0 |NULL|17.0|26.0|13.0|
|USW00014819|CHICAGO MIDWAY AIRPORT, IL US|2001-01-02|8.72 |0.0 |NULL|16.0|20.0|7.0 |
|USW00014819|CHICAGO MIDWAY AIRPORT, IL US|2001-01-03|12.08|0.0 |NULL|15.0|28.0|7.0 |
|USW00014819|CHICAGO MIDWAY AIRPORT, IL US|2001-01-04|12.08|0.0 |NULL|14.0|32.0|23.0|
|USW00014819|CHICAGO MIDWAY AIRPORT, IL US|2001-01-05|13.2 |0.0 |0.0 |13.0|37.0|22.0|
|USW00014819|CHICAGO MIDWAY AIRPORT, IL US|2001-01-06|9.4  |0.0 |0.0 |13.0|35.0|20.0|
|USW00014819|CHICAGO MIDWAY AIRPORT, IL US|2001-01-07|9.84 |0.0 |NULL|12.0|35.0|23.0|
|USW00014819|CHICAGO MIDWAY AIRPORT, IL US|2001-01-08|9.62 |0.0 |NULL|12.0|29.0|17.0|
|USW00014819|CHICAGO MIDWAY AIRPORT, IL US|2001-01-09|

### 14.2 Weather Source Validation

The weather data is validated to confirm that it represents the intended Chicago Midway Airport station and covers the requested analytical period.

The source date string is explicitly converted to a Spark date value for reliable temporal joins with daily crime aggregates.

In [39]:
weather_check_df = (
    weather_df
    .withColumn(
        "Weather_Date",
        F.to_date(F.col("DATE"), "yyyy-MM-dd")
    )
)

weather_check_df.agg(
    F.count("*").alias("Total_Weather_Records"),
    F.countDistinct("STATION").alias("Distinct_Stations"),
    F.min("Weather_Date").alias("Earliest_Weather_Date"),
    F.max("Weather_Date").alias("Latest_Weather_Date"),
    F.sum(
        F.col("Weather_Date").isNull().cast("long")
    ).alias("Invalid_Weather_Dates")
).show(truncate=False)

+---------------------+-----------------+---------------------+-------------------+---------------------+
|Total_Weather_Records|Distinct_Stations|Earliest_Weather_Date|Latest_Weather_Date|Invalid_Weather_Dates|
+---------------------+-----------------+---------------------+-------------------+---------------------+
|8398                 |1                |2001-01-01           |2023-12-31         |0                    |
+---------------------+-----------------+---------------------+-------------------+---------------------+



In [40]:
weather_check_df.select(
    "STATION",
    "NAME"
).distinct().show(truncate=False)

+-----------+-----------------------------+
|STATION    |NAME                         |
+-----------+-----------------------------+
|USW00014819|CHICAGO MIDWAY AIRPORT, IL US|
+-----------+-----------------------------+



### 14.3 Weather Missing-Value Profiling

Weather variables do not necessarily have complete observations for every day.

Missing values are therefore quantified before cleaning so that absent observations are not automatically confused with genuine zero measurements. This distinction is particularly important for snowfall and snow depth.

In [41]:
weather_measurements = [
    "AWND",
    "PRCP",
    "SNOW",
    "SNWD",
    "TMAX",
    "TMIN"
]

weather_null_profile = weather_check_df.agg(
    *[
        F.sum(
            F.col(column_name).isNull().cast("long")
        ).alias(column_name)
        for column_name in weather_measurements
    ]
)

weather_null_profile.show(truncate=False)

+----+----+----+----+----+----+
|AWND|PRCP|SNOW|SNWD|TMAX|TMIN|
+----+----+----+----+----+----+
|8   |16  |7995|7516|2   |2   |
+----+----+----+----+----+----+



In [42]:
weather_total_rows = weather_check_df.count()

weather_null_percentages = weather_check_df.agg(
    *[
        F.round(
            (
                F.sum(
                    F.col(column_name).isNull().cast("long")
                ) / F.lit(weather_total_rows)
            ) * 100,
            4
        ).alias(column_name)
        for column_name in weather_measurements
    ]
)

weather_null_percentages.show(truncate=False)

+------+------+-------+-------+------+------+
|AWND  |PRCP  |SNOW   |SNWD   |TMAX  |TMIN  |
+------+------+-------+-------+------+------+
|0.0953|0.1905|95.2012|89.4975|0.0238|0.0238|
+------+------+-------+-------+------+------+



### 14.4 Weather Date Continuity Assessment

Daily crime-weather analysis requires the weather series to contain one observation for each calendar date.

Although the source covers the complete 2001–2023 period, the observed row count is slightly lower than the number of calendar days in that interval. A complete date sequence is therefore generated and compared against the observed weather dates to identify any missing dates.

Missing dates are retained as an explicit data-quality issue rather than silently interpolated.

In [43]:
weather_date_range_df = (
    weather_check_df
    .agg(
        F.min("Weather_Date").alias("Start_Date"),
        F.max("Weather_Date").alias("End_Date")
    )
)

calendar_df = (
    weather_date_range_df
    .select(
        F.explode(
            F.sequence(
                F.col("Start_Date"),
                F.col("End_Date"),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("Weather_Date")
    )
)

expected_weather_days = calendar_df.count()

missing_weather_dates_df = (
    calendar_df
    .join(
        weather_check_df.select("Weather_Date"),
        on="Weather_Date",
        how="left_anti"
    )
    .orderBy("Weather_Date")
)

missing_weather_days = missing_weather_dates_df.count()

print(f"Expected calendar days : {expected_weather_days:,}")
print(f"Observed weather days  : {weather_total_rows:,}")
print(f"Missing calendar days  : {missing_weather_days:,}")
print(
    f"Date coverage          : "
    f"{(weather_total_rows / expected_weather_days) * 100:.4f}%"
)

Expected calendar days : 8,400
Observed weather days  : 8,398
Missing calendar days  : 2
Date coverage          : 99.9762%


In [44]:
missing_weather_dates_df.show(truncate=False)

+------------+
|Weather_Date|
+------------+
|2012-02-28  |
|2017-08-16  |
+------------+



## 15. Weather Data Cleaning and Feature Engineering

The weather dataset is highly complete for temperature, precipitation, and average wind speed, while snowfall and snow-depth measurements contain substantial missingness.

Missing snowfall and snow-depth observations are not automatically replaced with zero because an absent measurement does not necessarily indicate that no snow occurred. Preserving these null values avoids introducing unsupported assumptions into the analysis.

The weather date is converted to a Spark date type and several interpretable weather-event indicators are derived for later comparison with daily crime activity.

In [45]:
weather_clean_df = (
    weather_check_df
    .select(
        "STATION",
        "NAME",
        "Weather_Date",
        "AWND",
        "PRCP",
        "SNOW",
        "SNWD",
        "TMAX",
        "TMIN"
    )

    # Precipitation event
    .withColumn(
        "Has_Precipitation",
        F.when(
            F.col("PRCP").isNotNull() & (F.col("PRCP") > 0),
            F.lit(True)
        ).otherwise(F.lit(False))
    )

    # Snowfall event.
    # Null remains null because missing observation != zero snowfall.
    .withColumn(
        "Has_Snowfall",
        F.when(
            F.col("SNOW").isNull(),
            F.lit(None).cast("boolean")
        )
        .when(
            F.col("SNOW") > 0,
            F.lit(True)
        )
        .otherwise(F.lit(False))
    )

    # Mean daily temperature from observed maximum and minimum.
    .withColumn(
        "Mean_Temperature",
        F.when(
            F.col("TMAX").isNotNull() & F.col("TMIN").isNotNull(),
            (F.col("TMAX") + F.col("TMIN")) / 2
        )
    )

    # Daily temperature range.
    .withColumn(
        "Temperature_Range",
        F.when(
            F.col("TMAX").isNotNull() & F.col("TMIN").isNotNull(),
            F.col("TMAX") - F.col("TMIN")
        )
    )
)

### 15.1 Derived Weather Feature Verification

A sample of the transformed weather records is inspected to verify the derived precipitation, snowfall, and temperature features.

Weather measurements retain their original source units. Missing snowfall observations remain explicitly null rather than being converted to zero.

In [46]:
weather_clean_df.select(
    "Weather_Date",
    "AWND",
    "PRCP",
    "SNOW",
    "SNWD",
    "TMAX",
    "TMIN",
    "Mean_Temperature",
    "Temperature_Range",
    "Has_Precipitation",
    "Has_Snowfall"
).show(15, truncate=False)

+------------+-----+----+----+----+----+----+----------------+-----------------+-----------------+------------+
|Weather_Date|AWND |PRCP|SNOW|SNWD|TMAX|TMIN|Mean_Temperature|Temperature_Range|Has_Precipitation|Has_Snowfall|
+------------+-----+----+----+----+----+----+----------------+-----------------+-----------------+------------+
|2001-01-01  |7.38 |0.0 |NULL|17.0|26.0|13.0|19.5            |13.0             |false            |NULL        |
|2001-01-02  |8.72 |0.0 |NULL|16.0|20.0|7.0 |13.5            |13.0             |false            |NULL        |
|2001-01-03  |12.08|0.0 |NULL|15.0|28.0|7.0 |17.5            |21.0             |false            |NULL        |
|2001-01-04  |12.08|0.0 |NULL|14.0|32.0|23.0|27.5            |9.0              |false            |NULL        |
|2001-01-05  |13.2 |0.0 |0.0 |13.0|37.0|22.0|29.5            |15.0             |false            |false       |
|2001-01-06  |9.4  |0.0 |0.0 |13.0|35.0|20.0|27.5            |15.0             |false            |false 

### 15.2 Weather Event Distribution

The derived weather indicators are summarised to understand how frequently precipitation and observed snowfall events occur within the study period.

This provides a baseline for the later analysis of crime activity under different weather conditions.

In [47]:
weather_clean_df.groupBy(
    "Has_Precipitation"
).count().orderBy("Has_Precipitation").show()

+-----------------+-----+
|Has_Precipitation|count|
+-----------------+-----+
|            false| 5619|
|             true| 2779|
+-----------------+-----+



In [48]:
weather_clean_df.groupBy(
    "Has_Snowfall"
).count().orderBy("Has_Snowfall").show()

+------------+-----+
|Has_Snowfall|count|
+------------+-----+
|        NULL| 7995|
|       false|  372|
|        true|   31|
+------------+-----+



## 16. Police District Boundary Dataset

Official Chicago Police District boundary data is included to support the later comparison between data-driven crime clusters and administrative police districts.

The CSV source contains district identifiers together with polygon geometry stored as text. For Spark-based validation, the district identifiers are ingested and checked for completeness and uniqueness.

The corresponding GeoJSON file is retained for later spatial visualisation and boundary-overlay analysis.

In [49]:
police_schema = StructType([
    StructField("the_geom", StringType(), True),
    StructField("DIST_LABEL", StringType(), True),
    StructField("DIST_NUM", IntegerType(), True)
])

POLICE_CSV_PATH = RAW_DIR / "police_districts.csv"

police_df = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .schema(police_schema)
    .csv(str(POLICE_CSV_PATH))
)

print("Police District dataset loaded successfully.")
print("Number of columns:", len(police_df.columns))

Police District dataset loaded successfully.
Number of columns: 3


### 16.1 Police District Schema and Record Inspection

The police district source is inspected to confirm that district identifiers and geometry attributes have been loaded correctly.

In [50]:
police_df.printSchema()

root
 |-- the_geom: string (nullable = true)
 |-- DIST_LABEL: string (nullable = true)
 |-- DIST_NUM: integer (nullable = true)



In [51]:
police_df.select(
    "DIST_NUM",
    "DIST_LABEL"
).orderBy("DIST_NUM").show(30, truncate=False)

+--------+----------+
|DIST_NUM|DIST_LABEL|
+--------+----------+
|1       |1ST       |
|2       |2ND       |
|3       |3RD       |
|4       |4TH       |
|5       |5TH       |
|6       |6TH       |
|7       |7TH       |
|8       |8TH       |
|9       |9TH       |
|10      |10TH      |
|11      |11TH      |
|12      |12TH      |
|14      |14TH      |
|15      |15TH      |
|16      |16TH      |
|17      |17TH      |
|18      |18TH      |
|19      |19TH      |
|20      |20TH      |
|22      |22ND      |
|24      |24TH      |
|25      |25TH      |
|31      |31ST      |
|31      |31ST      |
|31      |31ST      |
+--------+----------+



### 16.2 Police District Identifier Validation

The district identifiers are checked for completeness, uniqueness, and range before the boundary data is used in later spatial analysis.

In [52]:
police_summary_df = police_df.agg(
    F.count("*").alias("Total_Records"),
    F.countDistinct("DIST_NUM").alias("Distinct_District_Numbers"),
    F.min("DIST_NUM").alias("Minimum_District_Number"),
    F.max("DIST_NUM").alias("Maximum_District_Number"),
    F.sum(
        F.col("DIST_NUM").isNull().cast("long")
    ).alias("Missing_District_Numbers")
)

police_summary_df.show(truncate=False)

+-------------+-------------------------+-----------------------+-----------------------+------------------------+
|Total_Records|Distinct_District_Numbers|Minimum_District_Number|Maximum_District_Number|Missing_District_Numbers|
+-------------+-------------------------+-----------------------+-----------------------+------------------------+
|25           |23                       |1                      |31                     |0                       |
+-------------+-------------------------+-----------------------+-----------------------+------------------------+



In [53]:
duplicate_police_districts_df = (
    police_df
    .groupBy("DIST_NUM")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_police_count = duplicate_police_districts_df.count()

print(
    f"Duplicated police district identifiers: "
    f"{duplicate_police_count}"
)

Duplicated police district identifiers: 1


### 16.3 Investigation of Repeated Police District Identifiers

The boundary source contains 25 geometry records but only 23 distinct district numbers. District `31` appears multiple times.

Because geographic boundary datasets may represent one administrative area using multiple polygon components, repeated district identifiers are not treated as duplicate errors automatically.

The repeated records are inspected before any transformation is applied so that legitimate spatial geometry is not accidentally discarded.

In [54]:
police_df.groupBy(
    "DIST_NUM",
    "DIST_LABEL"
).count().orderBy("DIST_NUM").show(30, truncate=False)

+--------+----------+-----+
|DIST_NUM|DIST_LABEL|count|
+--------+----------+-----+
|1       |1ST       |1    |
|2       |2ND       |1    |
|3       |3RD       |1    |
|4       |4TH       |1    |
|5       |5TH       |1    |
|6       |6TH       |1    |
|7       |7TH       |1    |
|8       |8TH       |1    |
|9       |9TH       |1    |
|10      |10TH      |1    |
|11      |11TH      |1    |
|12      |12TH      |1    |
|14      |14TH      |1    |
|15      |15TH      |1    |
|16      |16TH      |1    |
|17      |17TH      |1    |
|18      |18TH      |1    |
|19      |19TH      |1    |
|20      |20TH      |1    |
|22      |22ND      |1    |
|24      |24TH      |1    |
|25      |25TH      |1    |
|31      |31ST      |3    |
+--------+----------+-----+



In [55]:
police_df.filter(
    F.col("DIST_NUM") == 31
).select(
    "DIST_NUM",
    "DIST_LABEL",
    F.length("the_geom").alias("Geometry_Text_Length")
).show(truncate=False)

+--------+----------+--------------------+
|DIST_NUM|DIST_LABEL|Geometry_Text_Length|
+--------+----------+--------------------+
|31      |31ST      |8063                |
|31      |31ST      |34471               |
|31      |31ST      |1778                |
+--------+----------+--------------------+



### 16.4 Police District Lookup Construction

The complete boundary dataset is retained unchanged for later spatial analysis.

A separate compact lookup table containing one row per district is created for non-spatial reference purposes. This avoids discarding potentially legitimate multipart boundary geometry.

In [56]:
police_lookup_df = (
    police_df
    .select(
        F.col("DIST_NUM").alias("District"),
        F.trim(F.col("DIST_LABEL")).alias("District_Label")
    )
    .filter(F.col("District").isNotNull())
    .dropDuplicates(["District"])
    .orderBy("District")
)

print(
    "Unique police district lookup records:",
    police_lookup_df.count()
)

police_lookup_df.show(30, truncate=False)

Unique police district lookup records: 23
+--------+--------------+
|District|District_Label|
+--------+--------------+
|1       |1ST           |
|2       |2ND           |
|3       |3RD           |
|4       |4TH           |
|5       |5TH           |
|6       |6TH           |
|7       |7TH           |
|8       |8TH           |
|9       |9TH           |
|10      |10TH          |
|11      |11TH          |
|12      |12TH          |
|14      |14TH          |
|15      |15TH          |
|16      |16TH          |
|17      |17TH          |
|18      |18TH          |
|19      |19TH          |
|20      |20TH          |
|22      |22ND          |
|24      |24TH          |
|25      |25TH          |
|31      |31ST          |
+--------+--------------+



## 17. Final Crime Dataset Preparation

The cleaned crime dataset is prepared for persistent storage in Parquet format.

The final analytical dataset retains the original crime attributes together with parsed timestamp fields and derived temporal features.

The original `Year` and `District` fields are retained because they will be used as partition columns when writing the Parquet dataset.

No global removal of records with missing optional geographic attributes is performed. This preserves the maximum number of valid crime records while allowing task-specific subsets to be created later.

In [57]:
crime_final_df = (
    crime_clean_df
    .select(
        "ID",
        "Case Number",
        "Crime_Timestamp",
        "Crime_Date",
        "Block",
        "IUCR",
        "Primary Type",
        "Description",
        "Location Description",
        "Arrest",
        "Domestic",
        "Beat",
        "District",
        "Ward",
        "Community Area",
        "FBI Code",
        "X Coordinate",
        "Y Coordinate",
        "Year",
        "Updated_Timestamp",
        "Latitude",
        "Longitude",
        "Location",
        "Crime_Year",
        "Crime_Month",
        "Crime_Month_Name",
        "Crime_Day",
        "Crime_Day_Of_Week",
        "Crime_Hour",
        "Is_Weekend"
    )
)

print("Final crime dataset columns:", len(crime_final_df.columns))

Final crime dataset columns: 30


In [58]:
crime_final_df.printSchema()

root
 |-- ID: long (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Crime_Timestamp: timestamp (nullable = true)
 |-- Crime_Date: date (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community Area: integer (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: double (nullable = true)
 |-- Y Coordinate: double (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated_Timestamp: timestamp (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)
 |-- Crime_Year: integer (nullable = t

### 17.1 Final Crime Record Retention

The final analytical DataFrame is validated against the original raw record count to confirm that the cleaning process has not unintentionally removed valid crime records.

In [59]:
final_crime_count = crime_final_df.count()

retention_percentage = (
    final_crime_count / total_rows
) * 100

print(f"Raw crime records      : {total_rows:,}")
print(f"Final crime records    : {final_crime_count:,}")
print(f"Record retention       : {retention_percentage:.4f}%")

Raw crime records      : 7,784,664
Final crime records    : 7,784,664
Record retention       : 100.0000%


In [60]:
!python -m pip install pyspark==3.5.6 --no-cache-dir --default-timeout=1000 --retries 10


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [61]:
import pyspark
print("PySpark version:", pyspark.__version__)

PySpark version: 3.5.6


In [62]:
import shutil

print("winutils path:", shutil.which("winutils"))

winutils path: C:\hadoop\bin\winutils.EXE


In [63]:
import subprocess

result = subprocess.run(
    [r"C:\hadoop\bin\winutils.exe", "ls", "."],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

Return code: 0
drwx------ 1 DESKTOP-PAGHCSD\USER DESKTOP-PAGHCSD\None 4096 Sep 24 2026 .




In [67]:
from pathlib import Path

TEST_PARQUET_PATH = PROCESSED_DIR / "_parquet_write_test"

(
    crime_final_df
    .limit(1000)
    .write
    .mode("overwrite")
    .parquet(str(TEST_PARQUET_PATH))
)

print("Small Parquet write test passed.")

Small Parquet write test passed.


In [68]:
test_parquet_df = spark.read.parquet(
    str(TEST_PARQUET_PATH)
)

print("Test rows:", test_parquet_df.count())

Test rows: 1000


In [69]:
import shutil

if TEST_PARQUET_PATH.exists():
    shutil.rmtree(TEST_PARQUET_PATH)
    print(
        "Temporary Parquet write-test directory removed successfully."
    )
else:
    print(
        "Temporary Parquet write-test directory does not exist."
    )

Temporary Parquet write-test directory removed successfully.


## 18. Partitioned Parquet Storage

The final Chicago Crime dataset is persisted in Apache Parquet format.

Parquet is a columnar storage format that supports efficient compression, reduced disk I/O, predicate pushdown, and selective column access for analytical workloads.

The dataset is partitioned by `Year` and `District`. These attributes are commonly used for filtering and aggregation, so the partition structure enables Spark to prune irrelevant directories when queries target particular years or police districts.

This storage strategy prepares the processed crime dataset for the subsequent exploratory, clustering, and predictive modelling tasks.

In [70]:
CRIME_PARQUET_PATH = PROCESSED_DIR / "crime_parquet"

(
    crime_final_df
    .write
    .mode("overwrite")
    .partitionBy("Year", "District")
    .parquet(str(CRIME_PARQUET_PATH))
)

print("Partitioned crime Parquet dataset written successfully.")
print("Output path:", CRIME_PARQUET_PATH)

Partitioned crime Parquet dataset written successfully.
Output path: C:\Users\USER\Desktop\Masters\Big Enterprise\DAS7002\data\processed\crime_parquet


### 18.1 Parquet Read-Back Validation

The persisted Parquet dataset is reloaded independently to confirm that the output is readable and that the complete crime record population has been preserved.

In [71]:
crime_parquet_df = spark.read.parquet(
    str(CRIME_PARQUET_PATH)
)

parquet_row_count = crime_parquet_df.count()

print(f"Original final records : {final_crime_count:,}")
print(f"Parquet records        : {parquet_row_count:,}")
print(
    "Row-count match       :",
    parquet_row_count == final_crime_count
)

Original final records : 7,784,664
Parquet records        : 7,784,664
Row-count match       : True


### 18.2 Physical Partition Verification

The output directory is inspected to confirm that Spark has physically organised the Parquet dataset using the selected `Year` and `District` partition hierarchy.

In [72]:
year_partitions = sorted(
    [
        path.name
        for path in CRIME_PARQUET_PATH.iterdir()
        if path.is_dir() and path.name.startswith("Year=")
    ]
)

print("Number of year partitions:", len(year_partitions))
print("First year partitions:", year_partitions[:5])
print("Last year partitions :", year_partitions[-5:])

Number of year partitions: 23
First year partitions: ['Year=2001', 'Year=2002', 'Year=2003', 'Year=2004', 'Year=2005']
Last year partitions : ['Year=2019', 'Year=2020', 'Year=2021', 'Year=2022', 'Year=2023']


In [73]:
sample_year_path = CRIME_PARQUET_PATH / "Year=2023"

district_partitions = sorted(
    [
        path.name
        for path in sample_year_path.iterdir()
        if path.is_dir() and path.name.startswith("District=")
    ]
)

print("2023 district partitions:")
print(district_partitions)

2023 district partitions:
['District=1', 'District=10', 'District=11', 'District=12', 'District=14', 'District=15', 'District=16', 'District=17', 'District=18', 'District=19', 'District=2', 'District=20', 'District=22', 'District=24', 'District=25', 'District=3', 'District=31', 'District=4', 'District=5', 'District=6', 'District=7', 'District=8', 'District=9']


## 19. Persisting Cleaned Supporting Datasets

The cleaned supporting datasets are persisted in Parquet format to create a consistent processed-data layer for subsequent analytical tasks.

These datasets are substantially smaller than the Chicago Crime dataset and therefore do not require the same `Year` and `District` partitioning strategy. They are stored as independent Parquet datasets for efficient reuse during exploratory analysis, joins, clustering, and predictive modelling.

In [74]:
SOCIO_PARQUET_PATH = PROCESSED_DIR / "socioeconomic_parquet"
COMMUNITY_PARQUET_PATH = PROCESSED_DIR / "community_lookup_parquet"
WEATHER_PARQUET_PATH = PROCESSED_DIR / "weather_parquet"
POLICE_PARQUET_PATH = PROCESSED_DIR / "police_lookup_parquet"

socio_clean_df.write.mode("overwrite").parquet(
    str(SOCIO_PARQUET_PATH)
)

community_lookup_df.write.mode("overwrite").parquet(
    str(COMMUNITY_PARQUET_PATH)
)

weather_clean_df.write.mode("overwrite").parquet(
    str(WEATHER_PARQUET_PATH)
)

police_lookup_df.write.mode("overwrite").parquet(
    str(POLICE_PARQUET_PATH)
)

print("Supporting datasets written successfully.")
print("Socioeconomic :", SOCIO_PARQUET_PATH)
print("Community     :", COMMUNITY_PARQUET_PATH)
print("Weather       :", WEATHER_PARQUET_PATH)
print("Police        :", POLICE_PARQUET_PATH)

Supporting datasets written successfully.
Socioeconomic : C:\Users\USER\Desktop\Masters\Big Enterprise\DAS7002\data\processed\socioeconomic_parquet
Community     : C:\Users\USER\Desktop\Masters\Big Enterprise\DAS7002\data\processed\community_lookup_parquet
Weather       : C:\Users\USER\Desktop\Masters\Big Enterprise\DAS7002\data\processed\weather_parquet
Police        : C:\Users\USER\Desktop\Masters\Big Enterprise\DAS7002\data\processed\police_lookup_parquet


### 19.1 Supporting Dataset Validation

Each supporting Parquet dataset is reloaded and counted to verify that the persisted outputs preserve the expected number of cleaned records.

In [75]:
socio_check_df = spark.read.parquet(str(SOCIO_PARQUET_PATH))
community_check_df = spark.read.parquet(str(COMMUNITY_PARQUET_PATH))
weather_check_df = spark.read.parquet(str(WEATHER_PARQUET_PATH))
police_check_df = spark.read.parquet(str(POLICE_PARQUET_PATH))

print("Socioeconomic records :", socio_check_df.count())
print("Community records     :", community_check_df.count())
print("Weather records       :", weather_check_df.count())
print("Police lookup records :", police_check_df.count())

Socioeconomic records : 77
Community records     : 77
Weather records       : 8398
Police lookup records : 23


## 20. Final ETL Pipeline Validation

A final validation is performed to confirm that the distributed data-engineering pipeline has produced the expected analysis-ready outputs.

The validation summarises record preservation, Parquet persistence, partitioning, and the availability of the cleaned supporting datasets. These checks provide evidence that the processed data layer is ready for the exploratory analysis and machine-learning stages of the project.

In [76]:
print("=" * 60)
print("DAS7002 TASK 1 - FINAL ETL VALIDATION")
print("=" * 60)

print(f"Final crime records        : {final_crime_count:,}")
print(f"Parquet crime records      : {parquet_row_count:,}")
print(f"Parquet row-count match    : {parquet_row_count == final_crime_count}")
print(f"Year partitions            : {len(year_partitions)}")

print("-" * 60)

print(f"Socioeconomic records      : {socio_check_df.count():,}")
print(f"Community lookup records   : {community_check_df.count():,}")
print(f"Weather records            : {weather_check_df.count():,}")
print(f"Police lookup records      : {police_check_df.count():,}")

print("-" * 60)
print("ETL pipeline validation completed successfully.")

DAS7002 TASK 1 - FINAL ETL VALIDATION
Final crime records        : 7,784,664
Parquet crime records      : 7,784,664
Parquet row-count match    : True
Year partitions            : 23
------------------------------------------------------------
Socioeconomic records      : 77
Community lookup records   : 77
Weather records            : 8,398
Police lookup records      : 23
------------------------------------------------------------
ETL pipeline validation completed successfully.


### 20.1 Validation Interpretation

The final validation confirms that the ETL pipeline completed successfully and preserved the full analytical crime dataset.

The processed crime dataset contains 7,784,664 records, and the independently reloaded Parquet dataset contains the same number of records, confirming that no records were lost during persistence.

The dataset has been physically partitioned across 23 yearly partitions using `Year` and `District`, providing an efficient storage structure for subsequent temporal and district-level analytical queries.

The supporting socioeconomic, community-area, weather, and police-district lookup datasets were also successfully persisted and validated. Together, these outputs form the processed data layer that will be used in the exploratory analysis, clustering, and predictive modelling stages.